# LFM Instance Segmentation Example Workflow
This notebook is an example workflow of doing instance segmentation on visual light (RGB) bands of Lunar data. 

You can get started with this notebook by downloading it with:

```bash
wget https://raw.githubusercontent.com/nasa-nccs-hpda/lfm/refs/heads/main/notebooks/finetune_dinov3.ipynb
```

**See the README in the [repo](https://github.com/nasa-nccs-hpda/lfm)** for more info on how to use this notebook, and more on the process of training the model. 

## Imports, Dino Repo Clone

In [8]:
import os
import sys
import torch
import subprocess
from glob import glob

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

%matplotlib inline

In [9]:
# repo_dir = "lfm"

# if not os.path.exists(repo_dir):
#     subprocess.run(["git", "clone", f"https://github.com/nasa-nccs-hpda/{repo_dir}"])
# else:
#     subprocess.run(["git", "-C", repo_dir, "pull"])

In [10]:
sys.path.append("lfm")

from lfm.tasks.sem_segmentation.sseg_model import DINOSegmentation, load_dinov3_encoder
from lfm.tasks.sem_segmentation.sseg_dataset import get_dataloaders
from lfm.tasks.sem_segmentation.sseg_driver import train_model

## Main workflow

1. Define user-configured variables
2. Create dataloaders from files on /explore/nobackup/.
3. Load DinoV3 encoder, create encoder/decoder finetuning model.
4. Train model, print training stats, and visualize results. 

### User Config

In [11]:
# Local image of DinoV3 encoder
weights_local_checkpoint = '/explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'

# Data paths
INPUT_ROOT_DIR = "/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/7_band_vis_uv/sem_seg"
IMAGE_DIR = f"{INPUT_ROOT_DIR}/chips"  # Where input images are stored
LABEL_DIR = f"{INPUT_ROOT_DIR}/labels_npy"  # Where input labels are stored
IMAGE_FILE_TYPE = ".tif"
LABEL_FILE_TYPE = ".npy"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs/sem_seg"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Dataset parameters
MAX_SAMPLES = 500  # Set to None to use all available samples, or an integer to limit
TRAIN_SPLIT = 0.8  # 80% train, 20% validation

# Training hyperparameters
BATCH_SIZE = 16  # Number of images fed into the model at a time
NUM_EPOCHS = 50  # 10 is the default value, change to more epochs for more model learning
BASE_LR = 5e-5  # Starting learning rate
WEIGHT_DECAY = 1e-3  # Weight decay of optimizer
NUM_WORKERS = 8  # Used for parallelization
LOSS_TYPE = "focal_dice"  # Combined Dice + Binary CE loss
WARMUP_EPOCHS = 10  # Number of epochs for warmup LR scheduler
PATIENCE = 5  # How many epochs wait before early stopping

# Model parameters
TARGET_SIZE = (304, 304)  # Input size for DINO model
FREEZE_ENCODER = False
NUM_BANDS = 3  # Number of bands to use on the model; (3, 5, 7) are all supported
BAND_FILTER = [3, 1, 0]  # Bands to keep, in order of filtering

# Visualization and saving
CHECKPOINT_EVERY = 10  # Save checkpoint every N epochs
VISUALIZE_EVERY = 10  # Create visualizations every N epochs

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Output directory: ./outputs/sem_seg
Using device: cuda


### Create dataloaders

In [12]:
# ============================================================================
# CREATE DATALOADERS
# ============================================================================

print("\n" + "="*60)
print("STEP 1: Creating dataloaders.")
print("="*60)

train_loader, val_loader, MEAN, STD = get_dataloaders(
    image_dir=IMAGE_DIR,
    label_dir=LABEL_DIR,
    batch_size=BATCH_SIZE,
    train_split=TRAIN_SPLIT,
    num_workers=NUM_WORKERS,
    target_size=TARGET_SIZE,
    max_samples=MAX_SAMPLES,
    seed=42,
    stats_save_dir=OUTPUT_DIR,
    input_file_type=IMAGE_FILE_TYPE,
    label_file_type=LABEL_FILE_TYPE,
    debug=True,
    band_filter=BAND_FILTER
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")


STEP 1: Creating dataloaders.
Computing dataset statistics...


Computing dataset statistics:   0%|          | 2/674 [00:00<00:59, 11.31it/s]

First image shape: (300, 300, 7), dtype: float32
Detected 7 channel(s)
Value range BEFORE preprocessing: min=0.005635867360979319, max=0.16208191215991974
Value range AFTER min-max scaling: min=0.0, max=1.0


Computing dataset statistics: 100%|██████████| 674/674 [00:56<00:00, 11.84it/s]


Processed 674 valid images out of 674 total
Mean per channel: [0.3384354  0.35267626 0.3561474  0.35927844 0.36337809 0.3942253
 0.3927359 ]
Std per channel: [0.11590558 0.12117315 0.12044694 0.12096693 0.1202988  0.16930971
 0.16902104]

SANITY CHECKS:
✓ Min-max scaling was applied (expected for .tif)
✓ Mean should be ~0.3-0.7 (middle of [0,1] range)
✓ Std should be ~0.1-0.3
✓ Band 0: mean=0.3384, std=0.1159
✓ Band 1: mean=0.3527, std=0.1212
✓ Band 2: mean=0.3561, std=0.1204
✓ Band 3: mean=0.3593, std=0.1210
✓ Band 4: mean=0.3634, std=0.1203
✓ Band 5: mean=0.3942, std=0.1693
✓ Band 6: mean=0.3927, std=0.1690
✓ Saved statistics to ./outputs/sem_seg
Filtered inputs and mean/std to channels: [3, 1, 0]
Found 0 matched image-label pairs
Dataset configured for 3 channel(s)


ValueError: No matching image-label pairs found! Check that basenames match between image_dir and label_dir

### Load Encoder and Create Model

In [ ]:
print("\n" + "="*60)
print("STEP 2: Loading DINO encoder and creating model.")
print("="*60)

encoder = load_dinov3_encoder(weights_local_checkpoint, device=device)

In [ ]:
# Create model with DINO segmentation head, UNet decoder (see model.py)
print("Creating DINO segmentation model with UNet decoder...")
model = DINOSegmentation(
    encoder=encoder,  # Use DINOv3 head
    num_classes=2,  # Binary segmentaton (crater, not crater)
    img_size=TARGET_SIZE,
    num_bands=NUM_BANDS,  # Use flexible embeddings for 5/7-band vis data
).to(device)

# Unfreeze encoder for full fine-tuning if desired
if FREEZE_ENCODER:
    for param in model.encoder.parameters():
        param.requires_grad = False
    print("Encoder frozen (only decoder will be trained).")
else:
    print("Encoder unfrozen! Full model will be trained.")

### Run Training

In [ ]:
print("\n" + "="*60)
print("Starting training.")
print("="*60)

train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    mode="train",
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    learning_rate=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    checkpoint_every=CHECKPOINT_EVERY,
    visualize_every=VISUALIZE_EVERY,
    loss_type=LOSS_TYPE,
    device=device,
    warmup_epochs=WARMUP_EPOCHS,
    early_stopping_patience=PATIENCE,
    max_grad_norm=1.0
)

## Display some of the output visualizations

The training of the model is already producing some visualizations every N epochs.
Here we open some of the visualizations to look at them from the notebook.

In [ ]:
visualization_dir = os.path.join(OUTPUT_DIR, 'visualizations')
visualization_filenames = sorted(glob(os.path.join(visualization_dir, '*.png')))

In [ ]:
for vis_filename in visualization_filenames:
    img = mpimg.imread(vis_filename)
    plt.figure(figsize=(16, 14))
    plt.imshow(img)
    plt.axis("off")
    plt.show()